# LeetCode 930 — Binary Subarrays With Sum bằng Python

Link đề: https://leetcode.com/problems/binary-subarrays-with-sum/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), Sliding Window (cửa sổ trượt), Prefix Sum (tổng tiền tố)

Skeleton LeetCode:

```python
class Solution:
    def numSubarraysWithSum(self, nums: List[int], goal: int) -> int:
        ...
```

Ghi chú: đếm số **mảng con liên tiếp khác rỗng** có tổng bằng `goal` trong mảng nhị phân. Có ba cách tư duy kinh điển: **tổng tiền tố + bảng đếm** (`sum(l..r) = P[r+1] - P[l]`), **cửa sổ trượt "tối đa"** (`atMost(goal) - atMost(goal - 1)`) và **đếm theo vị trí các số `1`**. Cạm bẫy: cửa sổ trượt không áp dụng trực tiếp cho tổng **chính xác** vì các số `0` ở hai đầu tạo ra nhiều mảng con cùng tổng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm mảng con liên tiếp (không rỗng) của mảng `0/1` có tổng đúng bằng `goal`.
2. Nắm các công cụ Python cần thiết: tổng tiền tố, `collections.Counter`/`defaultdict`, mảng đếm theo giá trị tổng tiền tố (vì tổng `<= n`), hai con trỏ `left`/`right`, và danh sách vị trí các số `1`.
3. Hiểu nguyên lý: mảng con `nums[l..r]` có tổng `goal` ⇔ `P[r+1] - P[l] = goal`; và `đúng = atMost(goal) - atMost(goal - 1)`.
4. Đi qua 4 cách: thử mọi mảng con với tổng chạy (brute), tổng tiền tố + bảng đếm (cách nộp chính), cửa sổ trượt `atMost`, và đếm theo vị trí các số `1`; đối chiếu bằng oracle độc lập.
5. Tránh các lỗi kinh điển: dùng cửa sổ trượt cho tổng chính xác, quên `atMost(-1)` khi `goal = 0`, quên khởi tạo `{0: 1}` cho bảng tiền tố, và cập nhật bảng trước khi truy vấn (đếm cả mảng rỗng).

## 2. Đề bài & ví dụ

Cho mảng nhị phân `nums` và số nguyên `goal`, trả về **số lượng mảng con khác rỗng** có tổng bằng `goal`.

Mảng con là một phần **liên tiếp** của mảng.

**Ví dụ 1**

- Input: `nums = [1,0,1,0,1]`, `goal = 2`
- Output: `4`
- Giải thích: `[1,0,1]`, `[1,0,1,0]`, `[0,1,0,1]`, `[1,0,1]` (ở cuối).

**Ví dụ 2**

- Input: `nums = [0,0,0,0,0]`, `goal = 0`
- Output: `15` (số mảng con khác rỗng của mảng độ dài 5: `5 * 6 / 2`)

**Ràng buộc:**

- `1 <= nums.length <= 3 * 10^4`
- `nums[i]` là `0` hoặc `1`.
- `0 <= goal <= nums.length`

In [1]:
from typing import List
from collections import Counter, defaultdict
import random, time

examples = [
    ([1, 0, 1, 0, 1], 2, 4),
    ([0, 0, 0, 0, 0], 0, 15),
]
for nums, goal, expected in examples:
    print(f"nums={nums} goal={goal} kỳ vọng={expected}")

nums=[1, 0, 1, 0, 1] goal=2 kỳ vọng=4
nums=[0, 0, 0, 0, 0] goal=0 kỳ vọng=15


## 3. Kiến thức nền

### 3.1. Tổng tiền tố

`P[0] = 0`, `P[i + 1] = P[i] + nums[i]`. Tổng đoạn `nums[l..r]` là `P[r + 1] - P[l]` (với `0 <= l <= r < n`).

In [2]:
def prefix(nums: List[int]) -> List[int]:
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    return P

nums = [1, 0, 1, 0, 1]
P = prefix(nums)
print(P, "| tổng nums[1..3] =", P[4] - P[1])

[0, 1, 1, 2, 2, 3] | tổng nums[1..3] = 1


### 3.2. Đếm cặp `(l, r)` qua bảng đếm

Số mảng con có tổng `goal` = số cặp chỉ số `(i, j)` với `i < j` và `P[j] - P[i] = goal`. Duyệt `j` tăng dần, giữ `cnt[v]` = số lần giá trị tiền tố `v` đã xuất hiện ở các chỉ số `i < j`; cộng `cnt[P[j] - goal]`. Phải khởi tạo `cnt[0] = 1` (tiền tố rỗng `P[0]`).

### 3.3. Vì sao cửa sổ trượt phải dùng "tối đa"

Với mảng không âm, `atMost(k)` = số mảng con có tổng `<= k` tính được bằng hai con trỏ vì khi mở rộng phải, tổng chỉ tăng; khi thu trái, tổng chỉ giảm (tính **đơn điệu**). Tổng **chính xác** thì không đơn điệu theo cách thu hẹp: gặp `0` ở đầu, thu trái không đổi tổng nên có nhiều cửa sổ hợp lệ cùng một `right`. Giải pháp: `đúng(goal) = atMost(goal) - atMost(goal - 1)`.

In [3]:
def at_most(nums: List[int], k: int) -> int:
    if k < 0:
        return 0
    left = total = count = 0
    for right, x in enumerate(nums):
        total += x
        while total > k:
            total -= nums[left]
            left += 1
        count += right - left + 1                       # mọi mảng con kết thúc ở right bắt đầu từ left..right
    return count

nums = [1, 0, 1, 0, 1]
print("atMost(2) =", at_most(nums, 2), "| atMost(1) =", at_most(nums, 1), "| hiệu =", at_most(nums, 2) - at_most(nums, 1))

atMost(2) = 14 | atMost(1) = 10 | hiệu = 4


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `nums = [1,0,1,0,1]`, `goal = 2`. Tiền tố `P = [0,1,1,2,2,3]`. Với mỗi `j = 1..5`, đếm `i < j` có `P[i] = P[j] - 2`:

| `j` | `P[j]` | cần `P[i]` | số `i` thỏa |
|---|---|---|---|
| 1 | 1 | -1 | 0 |
| 2 | 1 | -1 | 0 |
| 3 | 2 | 0 | 1 (`i = 0`) |
| 4 | 2 | 0 | 1 (`i = 0`) |
| 5 | 3 | 1 | 2 (`i = 1, 2`) |

Tổng `0 + 0 + 1 + 1 + 2 = 4`. Đúng ví dụ.

## 5. Cách 1 — Brute Force: thử mọi mảng con

**Ý tưởng.** Với mỗi điểm bắt đầu `l`, mở rộng `r` và cộng dồn tổng; mỗi khi tổng bằng `goal` thì đếm. `O(n^2)`. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [4]:
def count_brute(nums: List[int], goal: int) -> int:
    n = len(nums)
    count = 0
    for l in range(n):
        total = 0
        for r in range(l, n):
            total += nums[r]
            if total == goal:
                count += 1
    return count

for nums, goal, expected in examples:
    got = count_brute(nums, goal)
    print(nums, goal, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 0, 1, 0, 1] 2 -> 4 OK
[0, 0, 0, 0, 0] 0 -> 15 OK


**Độ phức tạp và nút thắt.** `O(n^2)`: với `n = 3 * 10^4` là `4,5 * 10^8` phép cộng, quá chậm trong Python. Xem thời gian:

In [5]:
rng = random.Random(930)
for n in (500, 1000, 2000, 4000):
    nums = [rng.randint(0, 1) for _ in range(n)]
    t0 = time.perf_counter()
    count_brute(nums, n // 4)
    print(f"n={n:5}: {1000 * (time.perf_counter() - t0):9.1f} ms")

n=  500:       4.6 ms
n= 1000:      17.4 ms
n= 2000:      73.9 ms


n= 4000:     304.3 ms


Thời gian tăng bốn lần khi `n` tăng gấp đôi. Nút thắt: ta tính lại tổng đoạn cho từng cặp `(l, r)`, trong khi tổng đoạn là **hiệu của hai tổng tiền tố**.

## 6. Ý tưởng thuật toán / bước đột phá

Tổng đoạn `nums[l..r] = P[r + 1] - P[l]` nên bài toán trở thành **đếm cặp tiền tố có hiệu bằng `goal`**: `O(n)` với bảng đếm. Hai cách khác khai thác tính chất nhị phân:

- **Cửa sổ trượt `atMost`:** `đúng(goal) = atMost(goal) - atMost(goal - 1)`.
- **Vị trí các số `1`:** mảng con có đúng `goal` số `1` được xác định bởi `goal` số `1` liên tiếp và số `0` kẹp hai bên (nhân `(0 trước + 1) * (0 sau + 1)`); `goal = 0` đếm các đoạn toàn `0`.

| Cách | Ý tưởng |
|---|---|
| 2 | Tổng tiền tố + bảng đếm (nộp) |
| 3 | Cửa sổ trượt `atMost(goal) - atMost(goal - 1)` |
| 4 | Đếm theo vị trí các số `1` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tổng tiền tố + bảng đếm (cách nộp chính)

**Giải thích từng đoạn.**

- Vì `nums[i]` là `0/1` nên tổng tiền tố nằm trong `[0, n]`: dùng **mảng đếm** `cnt` độ dài `n + 1` thay cho từ điển (nhanh hơn).
- Khởi tạo `cnt[0] = 1` (tiền tố rỗng) và `s = 0`.
- Với mỗi `x`: `s += x`; nếu `s - goal >= 0` thì `ans += cnt[s - goal]`; sau đó `cnt[s] += 1`.
- Thứ tự quan trọng: **truy vấn trước, cập nhật sau**; nếu cập nhật trước thì khi `goal = 0` sẽ đếm cả mảng con rỗng.

In [6]:
def count_prefix(nums: List[int], goal: int) -> int:
    n = len(nums)
    cnt = [0] * (n + 1)
    cnt[0] = 1
    s = ans = 0
    for x in nums:
        s += x
        if s - goal >= 0:
            ans += cnt[s - goal]
        cnt[s] += 1
    return ans

for nums, goal, expected in examples:
    got = count_prefix(nums, goal)
    print(nums, goal, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 0, 1, 0, 1] 2 -> 4 OK
[0, 0, 0, 0, 0] 0 -> 15 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def numSubarraysWithSum(self, nums: List[int], goal: int) -> int:
        cnt = [0] * (len(nums) + 1)
        cnt[0] = 1
        s = ans = 0
        for x in nums:
            s += x
            if s >= goal:
                ans += cnt[s - goal]
            cnt[s] += 1
        return ans

sol_prefix = Solution().numSubarraysWithSum
assert [sol_prefix(list(n), g) for n, g, _ in examples] == [e for _, _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n)` thời gian, `O(n)` bộ nhớ.

---

### 7.2. Cách 3 — Cửa sổ trượt `atMost`

**Ý tưởng khác.** `atMost(k)` đếm mảng con có tổng `<= k`: với mỗi `right`, thu `left` cho tới khi tổng `<= k`, rồi cộng `right - left + 1` (mọi mảng con kết thúc ở `right` bắt đầu từ `left..right`). Đáp án `atMost(goal) - atMost(goal - 1)`, với `atMost(-1) = 0`.

**Giải thích từng đoạn.** `k < 0` phải trả `0` ngay: với `k = -1` vòng `while total > k` sẽ thu `left` vượt biên (tổng không bao giờ `<= -1`).

In [8]:
def count_sliding(nums: List[int], goal: int) -> int:
    return at_most(nums, goal) - at_most(nums, goal - 1)

for nums, goal, expected in examples:
    got = count_sliding(nums, goal)
    print(nums, goal, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 0, 1, 0, 1] 2 -> 4 OK
[0, 0, 0, 0, 0] 0 -> 15 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def numSubarraysWithSum(self, nums: List[int], goal: int) -> int:
        def at_most(k: int) -> int:
            if k < 0:
                return 0
            left = total = count = 0
            for right, x in enumerate(nums):
                total += x
                while total > k:
                    total -= nums[left]
                    left += 1
                count += right - left + 1
            return count

        return at_most(goal) - at_most(goal - 1)

sol_sliding = Solution().numSubarraysWithSum
assert [sol_sliding(list(n), g) for n, g, _ in examples] == [e for _, _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n)` thời gian (mỗi con trỏ đi tối đa `n` bước, hai lần gọi), `O(1)` bộ nhớ phụ.

---

### 7.3. Cách 4 — Đếm theo vị trí các số `1`

**Ý tưởng khác.** Gọi `pos` là danh sách chỉ số các số `1`, thêm mốc `-1` ở đầu và `n` ở cuối: `pos = [-1] + ones + [n]`.

- Nếu `goal > 0`: chọn `goal` số `1` liên tiếp `pos[i..i+goal-1]` (với các chỉ số trong `1..len(ones)`); mảng con phải chứa đúng các số `1` này: điểm bắt đầu nằm trong `(pos[i-1], pos[i]]` có `pos[i] - pos[i-1]` lựa chọn, điểm kết thúc nằm trong `[pos[i+goal-1], pos[i+goal])` có `pos[i+goal] - pos[i+goal-1]` lựa chọn; nhân hai số.
- Nếu `goal = 0`: mỗi đoạn `0` liên tiếp dài `g` (giữa hai số `1` hoặc hai đầu) đóng góp `g * (g + 1) / 2` mảng con.

**Giải thích từng đoạn.** `pos[i] - pos[i - 1] - 1` là số `0` kẹp giữa hai số `1` liên tiếp; ở `goal = 0` ta dùng độ dài đoạn `0` đó.

In [10]:
def count_positions(nums: List[int], goal: int) -> int:
    n = len(nums)
    pos = [-1] + [i for i, x in enumerate(nums) if x == 1] + [n]
    m = len(pos) - 2                                     # số số 1
    if goal == 0:
        total = 0
        for a, b in zip(pos, pos[1:]):
            g = b - a - 1                                # độ dài đoạn toàn 0
            total += g * (g + 1) // 2
        return total
    total = 0
    for i in range(1, m - goal + 2):                     # cửa sổ gồm các số 1 thứ i .. i + goal - 1
        left_choices = pos[i] - pos[i - 1]
        right_choices = pos[i + goal] - pos[i + goal - 1]
        total += left_choices * right_choices
    return total

for nums, goal, expected in examples:
    got = count_positions(nums, goal)
    print(nums, goal, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 0, 1, 0, 1] 2 -> 4 OK
[0, 0, 0, 0, 0] 0 -> 15 OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def numSubarraysWithSum(self, nums: List[int], goal: int) -> int:
        n = len(nums)
        pos = [-1] + [i for i, x in enumerate(nums) if x == 1] + [n]
        m = len(pos) - 2
        total = 0
        if goal == 0:
            for a, b in zip(pos, pos[1:]):
                g = b - a - 1
                total += g * (g + 1) // 2
            return total
        for i in range(1, m - goal + 2):
            total += (pos[i] - pos[i - 1]) * (pos[i + goal] - pos[i + goal - 1])
        return total

sol_positions = Solution().numSubarraysWithSum
assert [sol_positions(list(n), g) for n, g, _ in examples] == [e for _, _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n)` thời gian, `O(m)` bộ nhớ với `m` là số số `1`.

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 1: `nums = [1,0,1,0,1]`, `goal = 2`

In [12]:
def trace_prefix(nums, goal):
    cnt = defaultdict(int)
    cnt[0] = 1
    s = ans = 0
    print(f"khởi tạo cnt={dict(cnt)}")
    for x in nums:
        s += x
        add = cnt[s - goal] if s - goal >= 0 else 0
        ans += add
        cnt[s] += 1
        print(f"x={x}: s={s}, cần tiền tố {s - goal}: +{add} -> ans={ans}; cnt={dict(cnt)}")
    return ans

assert trace_prefix([1, 0, 1, 0, 1], 2) == 4

khởi tạo cnt={0: 1}
x=1: s=1, cần tiền tố -1: +0 -> ans=0; cnt={0: 1, 1: 1}
x=0: s=1, cần tiền tố -1: +0 -> ans=0; cnt={0: 1, 1: 2}
x=1: s=2, cần tiền tố 0: +1 -> ans=1; cnt={0: 1, 1: 2, 2: 1}
x=0: s=2, cần tiền tố 0: +1 -> ans=2; cnt={0: 1, 1: 2, 2: 2}
x=1: s=3, cần tiền tố 1: +2 -> ans=4; cnt={0: 1, 1: 2, 2: 2, 3: 1}


### 8.2. Cách 3 (cửa sổ trượt) cho `atMost(2)` với `nums = [1,0,1,0,1]`

In [13]:
def trace_at_most(nums, k):
    left = total = count = 0
    for right, x in enumerate(nums):
        total += x
        while total > k:
            total -= nums[left]
            left += 1
        count += right - left + 1
        print(f"right={right}: cửa sổ [{left}..{right}] tổng {total}; +{right - left + 1} mảng con kết thúc ở {right} -> {count}")
    return count

assert trace_at_most([1, 0, 1, 0, 1], 2) == 14
assert trace_at_most([1, 0, 1, 0, 1], 1) == 10
print("hiệu:", 14 - 10)

right=0: cửa sổ [0..0] tổng 1; +1 mảng con kết thúc ở 0 -> 1
right=1: cửa sổ [0..1] tổng 1; +2 mảng con kết thúc ở 1 -> 3
right=2: cửa sổ [0..2] tổng 2; +3 mảng con kết thúc ở 2 -> 6
right=3: cửa sổ [0..3] tổng 2; +4 mảng con kết thúc ở 3 -> 10
right=4: cửa sổ [1..4] tổng 2; +4 mảng con kết thúc ở 4 -> 14
right=0: cửa sổ [0..0] tổng 1; +1 mảng con kết thúc ở 0 -> 1
right=1: cửa sổ [0..1] tổng 1; +2 mảng con kết thúc ở 1 -> 3
right=2: cửa sổ [1..2] tổng 1; +2 mảng con kết thúc ở 2 -> 5
right=3: cửa sổ [1..3] tổng 1; +3 mảng con kết thúc ở 3 -> 8
right=4: cửa sổ [3..4] tổng 1; +2 mảng con kết thúc ở 4 -> 10
hiệu: 4


### 8.3. Cách 4 (vị trí các số `1`) cho ví dụ 1

`pos = [-1, 0, 2, 4, 5]`, `goal = 2`: cửa sổ `i = 1` (số `1` thứ nhất và thứ hai): `(pos[1] - pos[0]) * (pos[3] - pos[2]) = 1 * 2 = 2`; cửa sổ `i = 2` (số `1` thứ hai và thứ ba): `(pos[2] - pos[1]) * (pos[4] - pos[3]) = 2 * 1 = 2`. Tổng `4`.

In [14]:
nums, goal = [1, 0, 1, 0, 1], 2
pos = [-1] + [i for i, x in enumerate(nums) if x] + [len(nums)]
print("pos =", pos)
tot = 0
for i in range(1, len(pos) - 2 - goal + 2):
    lc, rc = pos[i] - pos[i - 1], pos[i + goal] - pos[i + goal - 1]
    print(f"i={i}: {lc} * {rc} = {lc * rc}")
    tot += lc * rc
assert tot == 4 == count_positions(nums, goal)

pos = [-1, 0, 2, 4, 5]
i=1: 1 * 2 = 2
i=2: 2 * 1 = 2


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (tổng tiền tố + mảng đếm): `O(n)`, ngắn, và dùng được cả khi mảng có số âm (với bảng băm).

```python
class Solution:
    def numSubarraysWithSum(self, nums: List[int], goal: int) -> int:
        cnt = [0] * (len(nums) + 1)
        cnt[0] = 1
        s = ans = 0
        for x in nums:
            s += x
            if s >= goal:
                ans += cnt[s - goal]
            cnt[s] += 1
        return ans
```

In [15]:
class Solution:
    def numSubarraysWithSum(self, nums: List[int], goal: int) -> int:
        cnt = [0] * (len(nums) + 1)
        cnt[0] = 1
        s = ans = 0
        for x in nums:
            s += x
            if s >= goal:
                ans += cnt[s - goal]
            cnt[s] += 1
        return ans

sol = Solution()
for nums, goal, expected in examples:
    got = sol.numSubarraysWithSum(list(nums), goal)
    print(nums, goal, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 0, 1, 0, 1] 2 -> 4 OK
[0, 0, 0, 0, 0] 0 -> 15 OK


## 10. Vì sao đúng

**(a) Tổng tiền tố.** Mảng con `nums[l..r]` (`0 <= l <= r < n`) tương ứng một cặp `(l, r + 1)` với `l < r + 1`; tổng của nó là `P[r + 1] - P[l]`. Số mảng con có tổng `goal` bằng số cặp `(i, j)`, `0 <= i < j <= n`, với `P[j] - P[i] = goal`. Duyệt `j` tăng dần và truy vấn `cnt[P[j] - goal]` trên **các `i < j` đã được thêm** đếm đúng số cặp; thêm `P[0] = 0` ban đầu (tiền tố rỗng) cho phép mảng con bắt đầu ở chỉ số `0`. Truy vấn **trước** khi thêm `P[j]` bảo đảm `i < j` (không đếm mảng con rỗng).

**(b) `atMost`.** Với mỗi `right`, tập các `left` sao cho `sum(left..right) <= k` là một đoạn `[L(right), right]` (vì mảng không âm: tổng giảm khi `left` tăng) và `L(right)` không giảm theo `right`. Hai con trỏ tính được `L(right)`; có `right - L + 1` mảng con. Số mảng con có tổng **đúng** `goal` = (số có tổng `<= goal`) - (số có tổng `<= goal - 1`) = `atMost(goal) - atMost(goal - 1)` (tổng là số nguyên). Với `goal = 0`: `atMost(-1) = 0` (không có mảng con tổng âm).

**(c) Vị trí các số `1`.** Mảng con có đúng `goal > 0` số `1` chứa đúng `goal` số `1` liên tiếp `pos[i], ..., pos[i+goal-1]`; điểm bắt đầu `l` phải thỏa `pos[i-1] < l <= pos[i]` (không chứa số `1` thứ `i - 1`): `pos[i] - pos[i-1]` lựa chọn; điểm kết thúc `r` thỏa `pos[i+goal-1] <= r < pos[i+goal]`: `pos[i+goal] - pos[i+goal-1]` lựa chọn; hai lựa chọn độc lập nên nhân. Với `goal = 0` mảng con toàn `0` nằm trong một đoạn `0` cực đại độ dài `g` có `g(g + 1)/2` mảng con.

**Kiểm chứng bằng máy.** Oracle **độc lập**: thử mọi mảng con (`O(n^2)`, Cách 1) trên `n` tới vài nghìn; với `n = 3 * 10^4` so ba cách `O(n)` có nguyên lý khác nhau (tiền tố, cửa sổ trượt, vị trí số `1`) với nhau và với oracle trên mẫu con.

In [16]:
rng = random.Random(1930)
for _ in range(500):
    n = rng.randint(1, 14)
    p = rng.choice([0.1, 0.5, 0.9])
    nums = [1 if rng.random() < p else 0 for _ in range(n)]
    goal = rng.randint(0, n)
    b = count_brute(nums, goal)
    assert b == count_prefix(nums, goal) == count_sliding(nums, goal) == count_positions(nums, goal), (nums, goal)
print("thử mọi mảng con, tiền tố, cửa sổ trượt, vị trí số 1 khớp nhau trên 500 ca ngẫu nhiên")

thử mọi mảng con, tiền tố, cửa sổ trượt, vị trí số 1 khớp nhau trên 500 ca ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi mảng con | `O(n^2)` | `O(1)` |
| 2. Tiền tố + mảng đếm | `O(n)` | `O(n)` |
| 3. Cửa sổ trượt `atMost` | `O(n)` | `O(1)` |
| 4. Vị trí các số `1` | `O(n)` | `O(m)` |

Đo thử với `n = 3 * 10^4` (cỡ tối đa):

In [17]:
rng = random.Random(11)
n = 3 * 10 ** 4
nums = [rng.randint(0, 1) for _ in range(n)]
goal = 1000
ref = count_prefix(nums, goal)
for name, fn in (("Cách 2 tiền tố", sol_prefix), ("Cách 3 cửa sổ trượt", sol_sliding), ("Cách 4 vị trí số 1", sol_positions), ("nộp", Solution().numSubarraysWithSum)):
    t0 = time.perf_counter()
    got = fn(list(nums), goal)
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref, name
    print(f"{name:20} n={n}: {dt:8.2f} ms (đáp án {got})")
t0 = time.perf_counter()
count_brute(nums[:3000], goal)
print(f"{'Cách 1 (n = 3000)':20}: {1000 * (time.perf_counter() - t0):8.1f} ms")

Cách 2 tiền tố       n=30000:     2.82 ms (đáp án 55987)
Cách 3 cửa sổ trượt  n=30000:     9.49 ms (đáp án 55987)
Cách 4 vị trí số 1   n=30000:     3.43 ms (đáp án 55987)
nộp                  n=30000:     3.13 ms (đáp án 55987)


Cách 1 (n = 3000)   :    172.0 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [18]:
def solutions_list():
    return {
        "prefix": count_prefix,
        "sliding": count_sliding,
        "positions": count_positions,
        "sol_prefix": sol_prefix,
        "sol_sliding": sol_sliding,
        "sol_positions": sol_positions,
        "final": Solution().numSubarraysWithSum,
    }

def run_all(nums, goal, expected=None, brute_limit=3000):
    ref = count_brute(nums, goal) if len(nums) <= brute_limit else count_prefix(nums, goal)
    for name, fn in solutions_list().items():
        assert fn(list(nums), goal) == ref, (name, nums[:20], goal)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", nums, goal, expected, ref)
    return ref

# 12.1 ví dụ đề
for nums, goal, expected in examples:
    assert run_all(nums, goal, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([0], 0, 1),                               # một mảng con [0]
    ([1], 0, 0),                               # [1] có tổng 1
    ([1], 1, 1),                               # [1]
    ([0], 1, 0),                               # không có tổng 1
    ([1, 1, 1], 3, 1),                         # cả mảng
    ([1, 1, 1], 2, 2),                         # [1,1] hai vị trí
    ([1, 1, 1], 1, 3),                         # ba số 1 đơn lẻ
    ([1, 1, 1], 0, 0),                         # không có số 0
    ([0, 0, 0], 0, 6),                         # 3*4/2
    ([0, 0, 0], 1, 0),
    ([0, 1, 0], 1, 4),                         # [1], [0,1], [1,0], [0,1,0]
    ([0, 1, 0, 0, 1], 1, 9),                   # 2*3 + 3*1
    ([1, 0, 1, 0, 1], 0, 2),                   # hai [0] riêng lẻ
    ([1, 0, 1, 0, 1], 3, 1),                   # cả mảng
    ([1, 0, 1, 0, 1], 4, 0),                   # goal vượt tổng
    ([0] * 5 + [1] + [0] * 5, 1, 36),          # 6 * 6 lựa chọn đầu cuối
]
for nums, goal, expected in edge:
    assert run_all(nums, goal, expected) == expected, (nums, goal)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 16 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_exact_window(nums, goal):               # cửa sổ trượt đếm trực tiếp tổng chính xác
    left = total = count = 0
    for right, x in enumerate(nums):
        total += x
        while total > goal and left <= right:
            total -= nums[left]
            left += 1
        if total == goal:
            count += 1
    return count

def wrong_at_most_negative(nums, goal):           # không xử lý atMost(-1): vòng while thu left vượt biên mảng
    def at_most(k):
        left = total = count = 0
        for right, x in enumerate(nums):
            total += x
            while total > k:
                total -= nums[left]
                left += 1
            count += right - left + 1
        return count
    return at_most(goal) - at_most(goal - 1)

def wrong_no_init(nums, goal):                    # quên cnt[0] = 1
    cnt = [0] * (len(nums) + 1)
    s = ans = 0
    for x in nums:
        s += x
        if s >= goal:
            ans += cnt[s - goal]
        cnt[s] += 1
    return ans

def wrong_update_first(nums, goal):               # cập nhật cnt trước khi truy vấn (đếm cả mảng rỗng)
    cnt = [0] * (len(nums) + 1)
    cnt[0] = 1
    s = ans = 0
    for x in nums:
        s += x
        cnt[s] += 1
        if s >= goal:
            ans += cnt[s - goal]
    return ans

def attempt(fn, nums, goal):
    try:
        return fn(list(nums), goal)
    except Exception as e:
        return f"lỗi {type(e).__name__}"

def find_counter(wrong, tries=20000, seed=4):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 8)
        nums = [rng.randint(0, 1) for _ in range(n)]
        goal = rng.randint(0, n)
        if attempt(wrong, nums, goal) != count_brute(nums, goal):
            return nums, goal
    return None

counters = {}
for name, fn in (("cửa sổ trượt cho tổng chính xác", wrong_exact_window), ("atMost(-1) không xử lý", wrong_at_most_negative),
                 ("quên cnt[0] = 1", wrong_no_init), ("cập nhật trước truy vấn", wrong_update_first)):
    c = find_counter(fn)
    assert c is not None, name
    counters[name] = c
    nums, goal = c
    print(f"{name:34} nums={nums} goal={goal}: sai -> {attempt(fn, nums, goal)}, đúng -> {count_brute(nums, goal)}")
    run_all(nums, goal)
print("12.3 phản ví dụ: OK")

cửa sổ trượt cho tổng chính xác    nums=[1, 0, 1, 1] goal=1: sai -> 4, đúng -> 5
atMost(-1) không xử lý             nums=[0, 0, 1, 1, 0] goal=0: sai -> lỗi IndexError, đúng -> 4
quên cnt[0] = 1                    nums=[1, 0, 1, 1] goal=1: sai -> 3, đúng -> 5
cập nhật trước truy vấn            nums=[0, 0, 1, 1, 0] goal=0: sai -> 9, đúng -> 4
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên nhỏ: mọi cách so với thử mọi mảng con (n <= 14)
rng = random.Random(4930)
for _ in range(1500):
    n = rng.randint(1, 14)
    p = rng.choice([0.05, 0.3, 0.5, 0.95])
    nums = [1 if rng.random() < p else 0 for _ in range(n)]
    run_all(nums, rng.randint(0, n))
print("12.4 ngẫu nhiên n <= 14 (1500 ca): OK")

# 12.5 trung bình: n tới 400, mọi goal có thể (0..n) trên vài mảng
rng = random.Random(5930)
for _ in range(40):
    n = rng.randint(15, 120)
    p = rng.choice([0.1, 0.5, 0.9])
    nums = [1 if rng.random() < p else 0 for _ in range(n)]
    for goal in range(0, n + 1):
        run_all(nums, goal)
print("12.5 mọi goal trên mảng tới 120 phần tử: OK")

# 12.6 lớn n = 3 * 10^4 (đúng ràng buộc): ba cách O(n) khớp nhau; oracle thử mọi mảng con trên mẫu con 2000
rng = random.Random(6930)
n = 3 * 10 ** 4
shapes = {
    "ngẫu nhiên": [rng.randint(0, 1) for _ in range(n)],
    "toàn 0": [0] * n,
    "toàn 1": [1] * n,
    "ít số 1": [1 if rng.random() < 0.01 else 0 for _ in range(n)],
    "xen kẽ": [i % 2 for i in range(n)],
}
for name, nums in shapes.items():
    for goal in (0, 1, 2, n // 100, n // 2, n):
        ref = count_prefix(nums, goal)
        for fname, fn in solutions_list().items():
            assert fn(list(nums), goal) == ref, (name, goal, fname)
    sub = nums[:2000]
    for goal in (0, 1, 7, 500):
        assert Solution().numSubarraysWithSum(list(sub), goal) == count_brute(sub, goal), (name, goal)
    print(f"  {name:10} OK")
print("12.6 n = 3 * 10^4: OK; ví dụ toàn 0, goal 0:", Solution().numSubarraysWithSum([0] * n, 0), "= n(n+1)/2 =", n * (n + 1) // 2)
assert Solution().numSubarraysWithSum([0] * n, 0) == n * (n + 1) // 2

# 12.7 tính chất: tổng số mảng con theo mọi goal = n(n+1)/2; đảo mảng không đổi đáp án
rng = random.Random(7930)
for _ in range(200):
    n = rng.randint(1, 40)
    nums = [rng.randint(0, 1) for _ in range(n)]
    total = sum(Solution().numSubarraysWithSum(nums, g) for g in range(0, n + 1))
    assert total == n * (n + 1) // 2, (nums, total)
    g = rng.randint(0, n)
    assert Solution().numSubarraysWithSum(nums[::-1], g) == Solution().numSubarraysWithSum(nums, g)
print("12.7 tính chất (tổng theo mọi goal, đảo mảng): OK")

12.4 ngẫu nhiên n <= 14 (1500 ca): OK


12.5 mọi goal trên mảng tới 120 phần tử: OK


  ngẫu nhiên OK


  toàn 0     OK


  toàn 1     OK


  ít số 1    OK


  xen kẽ     OK
12.6 n = 3 * 10^4: OK; ví dụ toàn 0, goal 0: 450015000 = n(n+1)/2 = 450015000
12.7 tính chất (tổng theo mọi goal, đảo mảng): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cửa sổ trượt cho tổng **chính xác**

Mỗi `right` chỉ cho **một** cửa sổ `[left, right]` khi tổng chính xác bằng `goal`, nhưng có thể có **nhiều** `left` hợp lệ (do các số `0` đầu cửa sổ).

In [21]:
nums, goal = counters["cửa sổ trượt cho tổng chính xác"]
print("nums =", nums, "goal =", goal, "| sai ->", wrong_exact_window(nums, goal), "| đúng ->", Solution().numSubarraysWithSum(list(nums), goal))
assert wrong_exact_window(nums, goal) != Solution().numSubarraysWithSum(list(nums), goal)

nums = [1, 0, 1, 1] goal = 1 | sai -> 4 | đúng -> 5


Vì sao sai: với `[0, 1]`, `goal = 1`: `right = 1` có hai `left` hợp lệ (`0` và `1`) nhưng chỉ đếm một. **Đúng:** `atMost(goal) - atMost(goal - 1)`.

### Lỗi 2 — Không xử lý `atMost(-1)` khi `goal = 0`

`atMost(-1)` phải bằng `0`; vòng `while total > k` với `k = -1` không bao giờ dừng (tổng không thể nhỏ hơn hoặc bằng `-1`) nên `left` chạy vượt biên mảng.

In [22]:
nums, goal = counters["atMost(-1) không xử lý"]
print("nums =", nums, "goal =", goal, "| sai ->", attempt(wrong_at_most_negative, nums, goal), "| đúng ->", Solution().numSubarraysWithSum(list(nums), goal))
assert attempt(wrong_at_most_negative, nums, goal) != Solution().numSubarraysWithSum(list(nums), goal)

nums = [0, 0, 1, 1, 0] goal = 0 | sai -> lỗi IndexError | đúng -> 4


Vì sao sai: không có mảng con nào tổng âm nên `atMost(-1) = 0`; nhưng vòng `while total > -1` không bao giờ dừng nên `left` chạy vượt biên mảng và gây `IndexError`. **Đúng:** `if k < 0: return 0`.

### Lỗi 3 — Quên khởi tạo `cnt[0] = 1`

Tiền tố rỗng (tổng `0` trước phần tử đầu) cần được tính để đếm các mảng con bắt đầu ở chỉ số `0`.

In [23]:
nums, goal = counters["quên cnt[0] = 1"]
print("nums =", nums, "goal =", goal, "| sai ->", wrong_no_init(nums, goal), "| đúng ->", Solution().numSubarraysWithSum(list(nums), goal))
assert wrong_no_init(nums, goal) != Solution().numSubarraysWithSum(list(nums), goal)

nums = [1, 0, 1, 1] goal = 1 | sai -> 3 | đúng -> 5


Vì sao sai: ví dụ `[1]`, `goal = 1` cần cặp `(P[0], P[1])`. **Đúng:** `cnt[0] = 1`.

### Lỗi 4 — Cập nhật bảng đếm trước khi truy vấn

Khi `goal = 0`, `cnt[s - 0]` sẽ đếm chính tiền tố hiện tại vừa thêm: mảng con **rỗng**.

In [24]:
nums, goal = counters["cập nhật trước truy vấn"]
print("nums =", nums, "goal =", goal, "| sai ->", wrong_update_first(nums, goal), "| đúng ->", Solution().numSubarraysWithSum(list(nums), goal))
assert wrong_update_first(nums, goal) != Solution().numSubarraysWithSum(list(nums), goal)

nums = [0, 0, 1, 1, 0] goal = 0 | sai -> 9 | đúng -> 4


Vì sao sai: cặp `(j, j)` tương ứng mảng con rỗng không được tính. **Đúng:** truy vấn trước, cập nhật sau.

### Lỗi 5 — Thử mọi mảng con (`O(n^2)`)

Với `n = 3 * 10^4`, `4,5 * 10^8` phép: TLE (xem mục 5 và 11). **Đúng:** một trong ba cách `O(n)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đếm mảng con có tổng bằng `k`" ⇒ **tổng tiền tố + bảng đếm** (đúng cả với số âm).
- Mảng **không âm** (đặc biệt `0/1`) ⇒ có thêm **cửa sổ trượt** `atMost(k) - atMost(k - 1)`.
- Truy vấn **trước**, cập nhật **sau**; khởi tạo `cnt[0] = 1`.
- Với `0/1` có thể đếm theo **vị trí các số `1`**.
- Cùng họ: Subarray Sum Equals K (560), Count Number of Nice Subarrays (1248), Subarrays with K Different Integers (992), Max Consecutive Ones III (1004).
- Test nên có: `goal = 0`, toàn `0`, toàn `1`, `goal` vượt tổng, một phần tử, `n = 3 * 10^4`.

## 15. Biến thể

1. **Số nguyên không âm bất kỳ** (không chỉ `0/1`): cửa sổ trượt `atMost` vẫn đúng; mảng đếm tiền tố cần từ điển.
2. **Có số âm:** chỉ còn cách tiền tố + từ điển (Subarray Sum Equals K).
3. **Đếm mảng con có tổng trong `[lo, hi]`** (bài tập 5): `atMost(hi) - atMost(lo - 1)`.
4. Bài liên quan: 560, 1248.

Dưới đây là biến thể 2 (tiền tố + từ điển, mảng có số âm) kiểm bằng thử mọi mảng con.

In [25]:
def subarray_sum_equals_k(nums: List[int], k: int) -> int:
    cnt = Counter({0: 1})
    s = ans = 0
    for x in nums:
        s += x
        ans += cnt[s - k]
        cnt[s] += 1
    return ans

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(1, 12))]
    k = rng.randint(-4, 4)
    brute = sum(1 for l in range(len(nums)) for r in range(l, len(nums)) if sum(nums[l:r + 1]) == k)
    assert subarray_sum_equals_k(nums, k) == brute, (nums, k)
print("biến thể 2 khớp; ví dụ:", subarray_sum_equals_k([1, 1, 1], 2), subarray_sum_equals_k([1, -1, 0], 0))

biến thể 2 khớp; ví dụ: 2 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi mảng con | Tổng chạy | `O(n^2)` | `O(1)` | Oracle |
| 2. Tiền tố + đếm | `P[j] - P[i] = goal` | `O(n)` | `O(n)` | **Cách nộp chính**, tổng quát |
| 3. Cửa sổ trượt | `atMost(g) - atMost(g - 1)` | `O(n)` | `O(1)` | Mảng không âm |
| 4. Vị trí số `1` | Nhân số lựa chọn hai đầu | `O(n)` | `O(m)` | Chỉ cho `0/1` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(prefix([1, 0, 1, 0, 1]))
print(at_most([1, 0, 1, 0, 1], 1), at_most([1, 0, 1, 0, 1], -1))
print(count_brute([0, 0, 0], 0))
```

In [26]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [0, 1, 0, 0, 1]` và `goal = 1`: ghi số mảng con có tổng `1`.

In [27]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách tổng tiền tố với mảng đếm.

In [28]:
def exercise_3_prefix(nums: List[int], goal: int) -> int:
    cnt = [0] * (len(nums) + 1)
    cnt[0] = 1
    s = ans = 0
    for x in nums:
        s += x
        # TODO 1: nếu s >= goal thì cộng cnt[s - goal] vào ans
        # TODO 2: tăng cnt[s]
        pass
    return ans


# for nums, goal, expected in examples:
#     assert exercise_3_prefix(nums, goal) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_sliding(nums, goal)` bằng `atMost(goal) - atMost(goal - 1)` (nhớ `atMost(-1) = 0`).

In [29]:
def exercise_4_sliding(nums: List[int], goal: int) -> int:
    # TODO
    pass


# for nums, goal, expected in examples:
#     assert exercise_4_sliding(nums, goal) == expected

### Bài 5 — Mini challenge: đếm mảng con có tổng trong `[lo, hi]` (khó)

Viết `exercise_5_range(nums, lo, hi)` trả về số mảng con khác rỗng có tổng `s` thỏa `lo <= s <= hi` (với `0 <= lo <= hi`), trong `O(n)`.

```python
exercise_5_range([1, 0, 1, 0, 1], 1, 2)  -> 12
exercise_5_range([0, 0, 0], 0, 0)        -> 6
exercise_5_range([1, 1], 3, 5)           -> 0
```

In [30]:
def exercise_5_range(nums: List[int], lo: int, hi: int) -> int:
    # TODO
    pass


# assert exercise_5_range([1, 0, 1, 0, 1], 1, 2) == 12
# assert exercise_5_range([0, 0, 0], 0, 0) == 6
# assert exercise_5_range([1, 1], 3, 5) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `prefix` bắt đầu bằng `0`; `atMost(-1)` là `0`.
- Mức 2: `atMost([1,0,1,0,1], 1)` đếm mảng con có không quá một số `1`.
- Mức 3: `[0, 1, 1, 2, 2, 3]`, `10 0`, `6`.

**Bài 2**
- Mức 1: hai số `1` ở chỉ số `1` và `4`.
- Mức 2: số `1` thứ nhất: `2` điểm bắt đầu x `3` điểm kết thúc = `6`; số `1` thứ hai: `3` x `1` = `3`.
- Mức 3: `9`.

**Bài 3**
- Mức 1: `if s >= goal: ans += cnt[s - goal]`.
- Mức 2: truy vấn trước.
- Mức 3: rồi `cnt[s] += 1`.

**Bài 4**
- Mức 1: `at_most(k)` trả `0` nếu `k < 0`.
- Mức 2: hai con trỏ `left`, `right`, cộng `right - left + 1`.
- Mức 3: `return at_most(goal) - at_most(goal - 1)`.

**Bài 5**
- Mức 1: số mảng con có tổng `<= hi` trừ số mảng con có tổng `<= lo - 1`.
- Mức 2: dùng `at_most`.
- Mức 3: `at_most(nums, hi) - at_most(nums, lo - 1)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [31]:
print(prefix([1, 0, 1, 0, 1]))
print(at_most([1, 0, 1, 0, 1], 1), at_most([1, 0, 1, 0, 1], -1))
print(count_brute([0, 0, 0], 0))

assert prefix([1, 0, 1, 0, 1]) == [0, 1, 1, 2, 2, 3]
assert (at_most([1, 0, 1, 0, 1], 1), at_most([1, 0, 1, 0, 1], -1)) == (10, 0)
assert count_brute([0, 0, 0], 0) == 6

[0, 1, 1, 2, 2, 3]
10 0
6


### Đáp án Bài 2

Số `1` thứ nhất (chỉ số `1`): `2 * 3 = 6`; số `1` thứ hai (chỉ số `4`): `3 * 1 = 3`; tổng `9`.

In [32]:
hand_answer_solution = 9
assert Solution().numSubarraysWithSum([0, 1, 0, 0, 1], 1) == hand_answer_solution == count_brute([0, 1, 0, 0, 1], 1)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [33]:
def exercise_3_prefix_solution(nums: List[int], goal: int) -> int:
    cnt = [0] * (len(nums) + 1)
    cnt[0] = 1
    s = ans = 0
    for x in nums:
        s += x
        if s >= goal:
            ans += cnt[s - goal]
        cnt[s] += 1
    return ans

for nums, goal, expected in examples:
    assert exercise_3_prefix_solution(nums, goal) == expected
rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 14))]
    goal = rng.randint(0, len(nums))
    assert exercise_3_prefix_solution(nums, goal) == count_brute(nums, goal)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [34]:
def exercise_4_sliding_solution(nums: List[int], goal: int) -> int:
    def at_most_k(k: int) -> int:
        if k < 0:
            return 0
        left = total = count = 0
        for right, x in enumerate(nums):
            total += x
            while total > k:
                total -= nums[left]
                left += 1
            count += right - left + 1
        return count

    return at_most_k(goal) - at_most_k(goal - 1)

for nums, goal, expected in examples:
    assert exercise_4_sliding_solution(nums, goal) == expected
rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 14))]
    goal = rng.randint(0, len(nums))
    assert exercise_4_sliding_solution(nums, goal) == count_brute(nums, goal)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [35]:
def exercise_5_range_solution(nums: List[int], lo: int, hi: int) -> int:
    return at_most(nums, hi) - at_most(nums, lo - 1)

def range_brute(nums, lo, hi):
    return sum(1 for l in range(len(nums)) for r in range(l, len(nums)) if lo <= sum(nums[l:r + 1]) <= hi)

assert exercise_5_range_solution([1, 0, 1, 0, 1], 1, 2) == 12
assert exercise_5_range_solution([0, 0, 0], 0, 0) == 6
assert exercise_5_range_solution([1, 1], 3, 5) == 0
rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 12))]
    lo = rng.randint(0, 4)
    hi = lo + rng.randint(0, 4)
    assert exercise_5_range_solution(nums, lo, hi) == range_brute(nums, lo, hi), (nums, lo, hi)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tổng tiền tố + mảng đếm)

```python
class Solution:
    def numSubarraysWithSum(self, nums: List[int], goal: int) -> int:
        cnt = [0] * (len(nums) + 1)       # cnt[v] = số lần tổng tiền tố v đã xuất hiện
        cnt[0] = 1                        # tiền tố rỗng
        s = ans = 0
        for x in nums:
            s += x
            if s >= goal:
                ans += cnt[s - goal]      # truy vấn TRƯỚC
            cnt[s] += 1                   # cập nhật SAU
        return ans
```

Công thức nhớ nhanh: `sum(l..r) = P[r+1] - P[l]`; đếm cặp tiền tố hiệu `goal`; hoặc `atMost(goal) - atMost(goal - 1)` với `atMost(-1) = 0`.

### Checklist trước khi submit

- [ ] Khởi tạo `cnt[0] = 1`?
- [ ] Truy vấn trước, cập nhật sau (không đếm mảng con rỗng)?
- [ ] Nếu dùng cửa sổ trượt: `atMost(goal) - atMost(goal - 1)` và `atMost(-1) = 0`?
- [ ] Không dùng cửa sổ trượt trực tiếp cho tổng chính xác?
- [ ] Đã thử: `goal = 0`, toàn `0`, toàn `1`, `goal` vượt tổng, một phần tử, `n = 3 * 10^4`?
- [ ] Tên method đúng `numSubarraysWithSum(self, nums, goal)` và trả `int`?